In [84]:
import pandas as pd
import glob
import os
import urllib.request

folder_path = "../data/Premier_League/Not_merged"
url_live = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
live_file_path = os.path.join(folder_path, "E0_25_26_LIVE.csv")
urllib.request.urlretrieve(url_live, live_file_path)
files = glob.glob(folder_path + "*.csv")

file_pattern = os.path.join(folder_path, "*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} files")

Found 26 files


*TODO: POBIERANIE NIE TYLKO 2026 ALE WSZYSTKICH DANYCH Z LAT OD JAKICHS 2000 DO NAJNOWSZYCH*


In [85]:
core_columns =[
    'Div', 'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR',
    'Referee',
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC',
    'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

all_matches = []

for file in files:
    df = pd.read_csv(file, encoding='utf-8', on_bad_lines='skip')

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce', format='mixed')

    cols_to_keep = [col for col in core_columns if col in df.columns]
    df = df[cols_to_keep]
    all_matches.append(df)


In [86]:
master_db = pd.concat(all_matches, ignore_index=True)

In [87]:
master_db = master_db.dropna(subset=['HomeTeam', 'AwayTeam'])
master_db=master_db.sort_values(by='Date').reset_index(drop=True)

In [88]:
def get_season(date):
    if pd.isna(date):
        return "Unknown"
    year = date.year
    month = date.month
    if month >= 7:
        return f"{year}/{year+1}"
    else:
        return f"{year-1}/{year}"

master_db['Season'] = master_db['Date'].apply(get_season)

In [89]:
output_dir = "../data/Premier_League"
output_filename = os.path.join(output_dir, "PremierLeague_WszystkieSezony.csv")

master_db.to_csv(output_filename, index=False)

**DANE Z PREMIER LEAGUE ZOSTALY SCALONE, WSTEPNIE OBROBIONE I WRZUCONE DO JEDNEGO PLIKU**

In [90]:
df=master_db.copy()
df.isna().sum()

Div         0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
Season      0
dtype: int64

In [91]:
df = master_db.copy()
ftr_mapping = {'H': 2, 'D': 1, 'A': 0}
df['Target'] = df['FTR'].map(ftr_mapping)

In [92]:
def get_home_points(ftr):
    if ftr == 'H': return 3
    elif ftr == 'D': return 1
    else: return 0

def get_away_points(ftr):
    if ftr == 'A': return 3
    elif ftr == 'D': return 1
    else: return 0

In [93]:
df['HomePoints'] = df['FTR'].apply(get_home_points)
df['AwayPoints'] = df['FTR'].apply(get_away_points)

print(df[['Date', 'HomeTeam', 'AwayTeam','FTR', 'HomePoints', 'AwayPoints']].head(10))

        Date    HomeTeam       AwayTeam FTR  HomePoints  AwayPoints
0 2000-08-19       Leeds        Everton   H           3           0
1 2000-08-19   Liverpool       Bradford   H           3           0
2 2000-08-19  Sunderland        Arsenal   H           3           0
3 2000-08-19    Charlton       Man City   H           3           0
4 2000-08-19     Chelsea       West Ham   H           3           0
5 2000-08-19    Coventry  Middlesbrough   A           0           3
6 2000-08-19       Derby    Southampton   D           1           1
7 2000-08-19   Tottenham        Ipswich   H           3           0
8 2000-08-19   Leicester    Aston Villa   D           1           1
9 2000-08-20  Man United      Newcastle   H           3           0
